# Experimental Design Principles

## Overview

A well-designed experiment is worth far more than any analysis applied to a poorly designed one. The core principles — randomisation, replication, blocking, and stratification — are the structural guarantees that make causal inference possible.

**Core principles:**

| Principle | Purpose | Without it |
|---|---|---|
| **Randomisation** | Balances known and unknown confounders in expectation | Systematic bias; no valid causal claim |
| **Replication** | Estimates experimental error; enables inference | Cannot distinguish treatment effect from noise |
| **Blocking** | Removes known nuisance variation → increases precision | Wasted variance; lower power |
| **Stratification** | Ensures balance across important subgroups | Imbalance → confounding |

**Design types covered:**
- Completely Randomised Design (CRD)
- Randomised Complete Block Design (RCBD)
- Stratified randomisation
- Factorial design preview (see `factorial_designs.ipynb`)

---

## Setup

In [1]:
library(tidyverse)
library(ggplot2)
library(agricolae)     # design.rcbd(), design.crd()
library(lme4)
library(broom)
library(patchwork)

set.seed(42)

# ── Study context: 60 riparian sites assigned to 3 restoration treatments ─────
# Known nuisance: catchment (n=10 catchments, 6 sites each)
# Confounders to balance: site elevation (high/low)

n_sites      <- 60
n_catchments <- 10
treatments   <- c("control","partial_restore","full_restore")

site_data <- tibble(
  site       = 1:n_sites,
  catchment  = rep(1:n_catchments, each=6),
  elevation  = rep(c("low","high"), times=30),
  catchment_effect = rep(rnorm(n_catchments, 0, 3), each=6)
)

Warning message:
"package 'tidyverse' was built under R version 4.4.3"
Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
Warning message:
"package 'dplyr' was built under R version 4.4.3"
Warning message:
"package 'stringr' was built under R version 4.4.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
"package 'agricolae' was built under R version 4.4.3"
Warning message:
"package 'lme4' was built under R version 4.4

---

## CRD vs. RCBD: The Cost of Ignoring Blocks

In [2]:
# ── CRD: completely random treatment assignment ───────────────────────────────
site_data <- site_data %>%
  mutate(
    treatment_crd = sample(rep(treatments, length.out=n_sites)),
    # RCBD: within each catchment, each treatment appears exactly twice
    treatment_rcbd = c(replicate(
      n_catchments,
      sample(rep(treatments, each=2))
    ))
  )

# Check balance: how well did each design balance catchments?
cat("CRD — treatment counts per catchment (ideal = 2 each):\n")
print(table(site_data$catchment, site_data$treatment_crd))

cat("\nRCBD — treatment counts per catchment (guaranteed = 2 each):\n")
print(table(site_data$catchment, site_data$treatment_rcbd))

# ── Simulate outcome and compare analysis power ───────────────────────────────
trt_effect <- c(control=0, partial_restore=3, full_restore=6)

site_data <- site_data %>%
  mutate(
    richness_crd  = 18 + trt_effect[treatment_crd]  + catchment_effect + rnorm(n_sites,0,3),
    richness_rcbd = 18 + trt_effect[treatment_rcbd] + catchment_effect + rnorm(n_sites,0,3)
  )

# CRD analysis: ignores catchment → larger residual variance
crd_fit  <- lm(richness_crd  ~ treatment_crd,  data=site_data)
# RCBD analysis: accounts for catchment → smaller residuals
rcbd_fit <- lm(richness_rcbd ~ treatment_rcbd + factor(catchment), data=site_data)

cat(sprintf("\nCRD  residual SE: %.3f\n", summary(crd_fit)$sigma))
cat(sprintf("RCBD residual SE: %.3f (blocking removed catchment noise)\n",
            summary(rcbd_fit)$sigma))

CRD — treatment counts per catchment (ideal = 2 each):
    
     control full_restore partial_restore
  1        1            3               2
  2        3            1               2
  3        2            3               1
  4        3            2               1
  5        0            5               1
  6        2            0               4
  7        1            2               3
  8        3            1               2
  9        3            1               2
  10       2            2               2

RCBD — treatment counts per catchment (guaranteed = 2 each):
    
     control full_restore partial_restore
  1        2            2               2
  2        2            2               2
  3        2            2               2
  4        2            2               2
  5        2            2               2
  6        2            2               2
  7        2            2               2
  8        2            2               2
  9        2            2        

---

## Stratified Randomisation

In [3]:
# Stratified randomisation: ensure balance within strata BEFORE randomising
# Critical when strata strongly predict the outcome

# ── Unstratified randomisation: can produce imbalance by chance ───────────────
naive_assign <- site_data %>%
  mutate(treatment=sample(rep(c("A","B"), n_sites/2)))

cat("Unstratified: elevation distribution by treatment:\n")
print(table(naive_assign$elevation, naive_assign$treatment))

# ── Stratified: randomise within each elevation stratum ───────────────────────
stratified_assign <- site_data %>%
  group_by(elevation) %>%
  mutate(treatment=sample(rep(c("A","B"), n()/2))) %>%
  ungroup()

cat("\nStratified: elevation distribution by treatment (guaranteed balance):\n")
print(table(stratified_assign$elevation, stratified_assign$treatment))

# ── Why it matters: simulate imbalance causing confounding ────────────────────
# High elevation sites have lower baseline richness
confound_sim <- naive_assign %>%
  mutate(
    baseline = ifelse(elevation=="high", 14, 22),
    richness = baseline + ifelse(treatment=="B", 2, 0) + rnorm(n_sites, 0, 2)
  )

# If high-elevation sites happen to be over-represented in group B:
elev_bias <- mean(confound_sim$baseline[confound_sim$treatment=="B"]) -
             mean(confound_sim$baseline[confound_sim$treatment=="A"])
cat(sprintf("\nBaseline richness imbalance (B−A): %.2f (due to elevation confounding)\n",
            elev_bias))
cat("Stratification eliminates this imbalance by design.\n")

Unstratified: elevation distribution by treatment:
      
        A  B
  high 14 16
  low  16 14

Stratified: elevation distribution by treatment (guaranteed balance):
      
        A  B
  high 15 15
  low  15 15

Baseline richness imbalance (B−A): -0.53 (due to elevation confounding)
Stratification eliminates this imbalance by design.


---

## Verifying Randomisation: Balance Checks

In [4]:
# After randomisation: check that baseline characteristics are balanced
# (This is NOT a hypothesis test — it's a design check)

# Add some baseline covariates to check
site_data <- site_data %>%
  mutate(
    treatment  = stratified_assign$treatment,
    baseline_n = rnorm(n_sites, 18, 4),
    slope_pct  = abs(rnorm(n_sites, 10, 5)),
    upstream_ha= rexp(n_sites, 0.02)
  )

balance_check <- site_data %>%
  group_by(treatment) %>%
  summarise(
    n            = n(),
    baseline_mean = round(mean(baseline_n), 2),
    slope_mean   = round(mean(slope_pct), 2),
    upstream_mean = round(mean(upstream_ha), 1),
    pct_high_elev = round(mean(elevation=="high")*100, 1)
  )

cat("Balance check (baseline covariates by treatment):\n")
print(balance_check)
cat("\nNote: small differences are expected by chance — do NOT use p-values to\n")
cat("test balance; report standardised differences (SMD) instead.\n")

# Standardised mean difference for each covariate
smd <- function(x, g, ref="A") {
  m <- tapply(x, g, mean); s <- tapply(x, g, sd)
  (m[names(m)!=ref] - m[ref]) / sqrt(mean(s^2))
}
cat(sprintf("\nSMD baseline_n: %.3f | slope: %.3f (|SMD| < 0.1 = good balance)\n",
            smd(site_data$baseline_n, site_data$treatment),
            smd(site_data$slope_pct,  site_data$treatment)))

Balance check (baseline covariates by treatment):
# A tibble: 2 × 6
  treatment     n baseline_mean slope_mean upstream_mean pct_high_elev
  <chr>     <int>         <dbl>      <dbl>         <dbl>         <dbl>
1 A            30          17.5      10.4           56.2            50
2 B            30          18.7       8.96          59.2            50

Note: small differences are expected by chance — do NOT use p-values to
test balance; report standardised differences (SMD) instead.

SMD baseline_n: 0.277 | slope: -0.287 (|SMD| < 0.1 = good balance)


---

## Common Pitfalls

**1. Pseudoreplication: treating subsamples as independent replicates**  
If three water samples are taken from each of ten sites, and all 30 samples are analysed as 30 independent observations, the effective sample size is 10 — not 30. Subsamples within a site are not independent. The experimental unit is the site; subsamples are technical replicates. Use mixed models with site as a random effect, or average within sites before analysis.

**2. Testing baseline balance with hypothesis tests**  
A common Table 1 practice is to report p-values for baseline covariate differences between groups. This is logically flawed: in a randomised experiment, any imbalance is by definition due to chance, and the p-value tells you only about sample size, not about the severity of imbalance. Report standardised mean differences (SMD) instead — |SMD| > 0.1 is a practical threshold for concern.

**3. Ignoring blocking structure in the analysis**  
If the experiment was designed with blocks (catchments, batches, fields), the blocking structure must be accounted for in the analysis — as a fixed effect (`lm(y ~ treatment + block)`) or random effect (`lmer(y ~ treatment + (1|block))`). Analysing a blocked design as a CRD inflates residual variance, reduces power, and produces anticonservative confidence intervals.

**4. Confounding treatment with time or space**  
Assigning all control sites in one catchment and all treatment sites in another is not randomisation — it completely confounds treatment with catchment. Similarly, treating all sites in spring and controls in autumn confounds treatment with season. Randomisation must occur within the relevant spatial and temporal scales.

**5. Changing the randomisation scheme after seeing baseline data**  
Adjusting the treatment assignments after seeing baseline covariates — even to "improve balance" — invalidates the randomisation guarantees. If balance on key covariates is important, use pre-specified stratified or covariate-adaptive randomisation before any baseline data are examined.

---
*r_methods_library · Samantha McGarrigle · [github.com/samantha-mcgarrigle](https://github.com/samantha-mcgarrigle)*